In [ ]:
!pip install -q -U "transformers>=4.55" "trl>=0.21" "peft>=0.17" "bitsandbytes>=0.46" "accelerate>=1.0" "datasets>=3.0" scikit-learn scipy

In [ ]:
import os, json, random, gc, subprocess
import numpy as np, torch

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

def najdp(kand):
    for k in kand:
        if os.path.isdir(k):
            return k
    return None

DAN_DIR = najdp([
    "/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/data",
    "/kaggle/input/ml-olympiad-red/ml-olympiad-red-task/data",
    "data",
])
MTR_DIR = najdp([
    "/kaggle/input/datasets/olesyasycheva/ml-olympiad-red/ml-olympiad-red-task/metrics",
    "/kaggle/input/ml-olympiad-red/ml-olympiad-red-task/metrics",
    "metrics",
])
if DAN_DIR is None or MTR_DIR is None:
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/syeva1/tbank-red-level.git", "_repo"], check=True)
    DAN_DIR = DAN_DIR or "_repo/data"
    MTR_DIR = MTR_DIR or "_repo/metrics"

print("DAN_DIR:", DAN_DIR)
print("MTR_DIR:", MTR_DIR)

def grzjl(p):
    with open(p) as f:
        return [json.loads(l) for l in f]

In [ ]:
import torch
from transformers import (AutoModelForCausalLM, AutoModelForSequenceClassification,
                          AutoTokenizer, BitsAndBytesConfig)
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training

MODEL = "Qwen/Qwen3-4B-Instruct-2507"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

LORTGT = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

def lorcfg(tt="CAUSAL_LM"):
    return LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                      task_type=tt, target_modules=LORTGT)

def grzbaz():
    m = AutoModelForCausalLM.from_pretrained(
        MODEL, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16)
    m.config.use_cache = False
    return m

def osvob():
    gc.collect(); torch.cuda.empty_cache()

In [ ]:
import pickle, scipy.sparse as sp

with open(os.path.join(MTR_DIR, "style_clf.pkl"), "rb") as f:
    _stl = pickle.load(f)
_LOGREG = _stl["clf"]; _WVEC, _CVEC = _stl["vecs"]

def prost(txts):
    X = sp.hstack([_WVEC.transform(txts), _CVEC.transform(txts)]).tocsr()
    return _LOGREG.predict_proba(X)[:, 1]

@torch.no_grad()
def genr(mod, prom, mnt=256):
    kmsg = [{"role": "user", "content": prom}]
    inp = tok.apply_chat_template(
        kmsg, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(mod.device)
    vyh = mod.generate(**inp, max_new_tokens=mnt, do_sample=False, pad_token_id=tok.pad_token_id)
    return tok.decode(vyh[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

def ocenpr(mod, tst):
    mod.eval()
    genl = [genr(mod, t["prompt"]) for t in tst]
    return float(np.mean(prost(genl)))

def interv(zn, grn):
    for bukva, lo, hi in grn:
        if lo <= zn < hi:
            return bukva
    return "?"

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

ka = grzjl(os.path.join(DAN_DIR, "kid_adult.jsonl"))

sftnab = Dataset.from_list([{
    "prompt":     [{"role": "user", "content": r["prompt"]}],
    "completion": [{"role": "assistant", "content": r["kid"]}],
} for r in ka])

baz = prepare_model_for_kbit_training(grzbaz())
sftarg = SFTConfig(
    output_dir="sft", per_device_train_batch_size=2, gradient_accumulation_steps=8,
    num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03,
    logging_steps=10, save_strategy="no", fp16=True, max_length=1024, seed=SEED,
    report_to="none", gradient_checkpointing=True)
sft = SFTTrainer(model=baz, args=sftarg, train_dataset=sftnab,
                 processing_class=tok, peft_config=lorcfg())
sft.train()

In [ ]:
tststl = grzjl(os.path.join(DAN_DIR, "public_test_style.jsonl"))
GRNSTL = [("Б", 0.9, 1.0001), ("А", 0.7, 0.9), ("Г", 0.4, 0.7), ("В", 0.0, 0.4)]
ps = ocenpr(sft.model, tststl)
print(f"P_simple = {ps:.4f}  ->  {interv(ps, GRNSTL)}")